In [ ]:
# =========================================================
# PIXEL-WISE HEAT INDEX THRESHOLDS
# ASSAM | ERA5-LAND HOURLY
#
# BASELINE:
# 1990-2023
#
# HOURS:
# 06-12 UTC
# (~11:30 AM - 5:30 PM IST)
#
# METHOD:
# For each pixel:
#   Tmax hour within 06-12 UTC
#   Dewpoint from same Tmax hour
#   RH from Tmax/Td
#   Heat Index
#
# SEASONS:
#   JFM = Jan-Mar
#   AMJ = Apr-Jun
#   JAS = Jul-Sep
#   OND = Oct-Dec
#
# PERCENTILES:
#   P80
#   P88
#   P95
#   P99
#
# OUTPUT:
#   48 GeoTIFFs
#
# =========================================================

import ee
import geemap
import geopandas as gpd

# =========================================================
# INITIALIZE EARTH ENGINE
# =========================================================

ee.Authenticate()
ee.Initialize(project="areca-farm")

# =========================================================
# LOAD ASSAM GEOJSON
# =========================================================

geojson_path = "../../Maps/Geojson/assam_rc_2024-11.geojson"

gdf = gpd.read_file(geojson_path)

gdf["geometry"] = gdf.geometry.simplify(
    0.05,
    preserve_topology=True
)

assam = gdf.dissolve()

assam_ee = geemap.geopandas_to_ee(assam)

region = assam_ee.geometry()

# Bounding box reduces payload size
export_region = region.bounds()

print("Assam geometry loaded")

# =========================================================
# ERA5 LAND HOURLY
# =========================================================

era5 = (
    ee.ImageCollection("ECMWF/ERA5_LAND/HOURLY")
    .filterDate("1990-01-01", "2024-01-01")
    .filter(ee.Filter.calendarRange(6, 12, "hour"))
)

print("ERA5 loaded")

# =========================================================
# ADD TEMPERATURE BAND (°C)
# =========================================================

def add_temp(img):

    T = (
        img.select("temperature_2m")
        .subtract(273.15)
        .rename("T")
    )

    return img.addBands(T)

era5 = era5.map(add_temp)

# =========================================================
# DAILY HEAT INDEX USING HOTTEST HOUR
# =========================================================

def make_daily_hi(day):

    day = ee.Date(day)

    daily = era5.filterDate(
        day,
        day.advance(1, "day")
    )

    # Select hottest hour (pixel-wise)
    hottest = daily.qualityMosaic("T")

    T = hottest.select("T")

    Td = (
        hottest
        .select("dewpoint_temperature_2m")
        .subtract(273.15)
    )

    RH = Td.expression(
        """
        100 * (
            exp((17.625 * Td)/(243.04 + Td))
            /
            exp((17.625 * T)/(243.04 + T))
        )
        """,
        {
            "Td": Td,
            "T": T
        }
    )

    HI = T.expression(
        """
        -8.784695 +
        1.61139411*T +
        2.338549*RH -
        0.14611605*T*RH -
        0.012308094*T*T -
        0.016424828*RH*RH +
        0.002211732*T*T*RH +
        0.00072546*T*RH*RH -
        0.000003582*T*T*RH*RH
        """,
        {
            "T": T,
            "RH": RH
        }
    ).rename("HI")

    return HI.set(
        "system:time_start",
        day.millis()
    )


# =========================================================
# MONTH DEFINITIONS
# =========================================================

months = {
    "JAN": 1,
    "FEB": 2,
    "MAR": 3,
    "APR": 4,
    "MAY": 5,
    "JUN": 6,
    "JUL": 7,
    "AUG": 8,
    "SEP": 9,
    "OCT": 10,
    "NOV": 11,
    "DEC": 12
}

percentiles = [80, 88, 95, 99]

# =========================================================
# BUILD DAILY COLLECTION FOR A MONTH
# =========================================================

def build_month_collection(target_month):

    start = ee.Date("1990-01-01")
    end = ee.Date("2024-01-01")

    n_days = end.difference(start, "day")

    days = ee.List.sequence(
        0,
        n_days.subtract(1)
    )

    def process_day(offset):

        day = start.advance(offset, "day")

        month = ee.Number.parse(
            day.format("M")
        )

        valid = month.eq(target_month)

        return ee.Image(
            ee.Algorithms.If(
                valid,
                make_daily_hi(day),
                None
            )
        )

    return ee.ImageCollection(
        days.map(process_day)
    )

# =========================================================
# EXPORT MONTHLY PERCENTILES
# =========================================================

for month_name, month_num in months.items():

    print(f"\nBuilding {month_name}")

    daily_hi = build_month_collection(
        month_num
    )

    for p in percentiles:

        print(
            f"Submitting {month_name} P{p}"
        )

        threshold = (
            daily_hi
            .reduce(
                ee.Reducer.percentile([p])
            )
            .rename(
                f"HI_P{p}"
            )
            .clip(region)
        )

        task = ee.batch.Export.image.toDrive(
            image=threshold,
            description=f"{month_name}_P{p}_1990_2023",
            folder="Monthly_Heat_Index_Percentiles_Assam",
            fileNamePrefix=f"{month_name}_P{p}_1990_2023",
            region=export_region,
            scale=9000,
            maxPixels=1e13
        )

        task.start()

print("\n====================================")
print("ALL EXPORTS SUBMITTED")
print("====================================")

print("\nExpected outputs:\n")

for month_name in months:

    for p in percentiles:

        print(
            f"{month_name}_P{p}_1990_2023.tif"
        )

Assam geometry loaded
ERA5 loaded

Building JAN
Submitting JAN P80
Submitting JAN P88
Submitting JAN P95
Submitting JAN P99

Building FEB
Submitting FEB P80
Submitting FEB P88
Submitting FEB P95
Submitting FEB P99

Building MAR
Submitting MAR P80
Submitting MAR P88
Submitting MAR P95
Submitting MAR P99

Building APR
Submitting APR P80
Submitting APR P88
Submitting APR P95
Submitting APR P99

Building MAY
Submitting MAY P80
Submitting MAY P88
Submitting MAY P95
Submitting MAY P99

Building JUN
Submitting JUN P80
Submitting JUN P88
Submitting JUN P95
Submitting JUN P99

Building JUL
Submitting JUL P80
Submitting JUL P88
Submitting JUL P95
Submitting JUL P99

Building AUG
Submitting AUG P80
Submitting AUG P88
Submitting AUG P95
Submitting AUG P99

Building SEP
Submitting SEP P80
Submitting SEP P88
Submitting SEP P95
Submitting SEP P99

Building OCT
Submitting OCT P80
Submitting OCT P88
Submitting OCT P95
Submitting OCT P99

Building NOV
Submitting NOV P80
Submitting NOV P88
Submitting NOV

In [ ]:
# random stuff below

In [ ]:
# =========================================================
# DAILY ERA5 HEAT INDEX EXTRACTION
# 11 AM – 5 PM | APRIL 2026
# DISTRICT-WISE DAILY DOWNLOAD
# =========================================================

# =========================================================
# 1. INSTALL PACKAGES
# =========================================================

# Run once in VSCode terminal:
# pip install earthengine-api geemap geopandas pandas numpy openpyxl


# =========================================================
# 2. IMPORTS
# =========================================================

import ee
import geemap
import geopandas as gpd
import pandas as pd
import numpy as np
from datetime import datetime, timedelta


# =========================================================
# 3. INITIALIZE EARTH ENGINE
# =========================================================

ee.Authenticate()
ee.Initialize(project='areca-farm')


# =========================================================
# 4. LOAD GEOJSON
# =========================================================

geojson_path = "../../Maps/Geojson/assam_rc_2025-04.geojson"

gdf = gpd.read_file(geojson_path)

print("GeoJSON Loaded")


# =========================================================
# 5. REMOVE COMPLEX BOUNDARIES
# =========================================================
# Helps avoid incomplete edge pixels
# and speeds up computation
# =========================================================

gdf["geometry"] = gdf.geometry.simplify(0.01)

print("Geometry Simplified")


# =========================================================
# 6. CONVERT TO EARTH ENGINE
# =========================================================

districts = geemap.geopandas_to_ee(gdf)


# =========================================================
# 7. DATE RANGE
# =========================================================

start_date = datetime(2021, 1, 1)
end_date   = datetime(2025, 1, 1)


# =========================================================
# 8. ERA5 COLLECTION
# =========================================================

era5 = ee.ImageCollection("ECMWF/ERA5_LAND/HOURLY")


# =========================================================
# 9. HEAT INDEX FUNCTION
# =========================================================

def add_heat_index(img):

    # Temperature (°C)
    t = img.select('temperature_2m').subtract(273.15)

    # Dewpoint (°C)
    d = img.select('dewpoint_temperature_2m').subtract(273.15)

    # Relative Humidity
    rh = (
        d.expression(
            '''
            100 * (
                exp((17.625 * Td)/(243.04 + Td)) /
                exp((17.625 * T)/(243.04 + T))
            )
            ''',
            {
                'Td': d,
                'T': t
            }
        )
    )

    # Heat Index
    hi = (
        t.expression(
            '''
            -8.784695 +
            1.61139411*T +
            2.338549*RH -
            0.14611605*T*RH -
            0.012308094*(T**2) -
            0.016424828*(RH**2) +
            0.002211732*(T**2)*RH +
            0.00072546*T*(RH**2) -
            0.000003582*(T**2)*(RH**2)
            ''',
            {
                'T': t,
                'RH': rh
            }
        )
    ).rename('HI')

    return hi.copyProperties(img, ['system:time_start'])


# =========================================================
# 10. DAILY LOOP
# =========================================================

all_days = []

current = start_date

while current < end_date:

    next_day = current + timedelta(days=1)

    print(f"\nProcessing: {current.strftime('%Y-%m-%d')}")

    # -----------------------------------------------------
    # FILTER DAILY DATA
    # -----------------------------------------------------

    daily = (
        era5
        .filterDate(
            current.strftime('%Y-%m-%d'),
            next_day.strftime('%Y-%m-%d')
        )
        .filter(ee.Filter.calendarRange(11, 16, 'hour'))
        .map(add_heat_index)
    )

    # -----------------------------------------------------
    # DAILY 11AM–5PM MEAN
    # -----------------------------------------------------

    daily_hi = daily.mean()

    # -----------------------------------------------------
    # DISTRICT MEAN
    # -----------------------------------------------------

    stats = daily_hi.reduceRegions(
        collection=districts,
        reducer=ee.Reducer.mean(),
        scale=11132,
        tileScale=4
    )

    # -----------------------------------------------------
    # DOWNLOAD SMALL DAILY TABLE
    # -----------------------------------------------------

    features = stats.getInfo()['features']

    rows = []

    for f in features:

        props = f['properties']

        rows.append({
            'date': current.strftime('%Y-%m-%d'),
            'district': props.get('dtname'),
            'heat_index': props.get('mean')
        })

    df_day = pd.DataFrame(rows)

    print("Collected Successfully")

    all_days.append(df_day)

    current = next_day


# =========================================================
# 11. COMBINE ALL DAYS
# =========================================================

daily_df = pd.concat(all_days, ignore_index=True)

print("\nDaily Collection Complete")


# =========================================================
# 12. MONTHLY DISTRICT MEAN
# =========================================================

monthly_df = (
    daily_df
    .groupby('district', as_index=False)
    ['heat_index']
    .mean()
)

monthly_df.rename(
    columns={'heat_index': 'April_2026_HI'},
    inplace=True
)

print("\nMonthly Means Computed")


# =========================================================
# 13. MERGE BACK TO GEOJSON ATTRIBUTES
# =========================================================

final_df = gdf.merge(
    monthly_df,
    left_on='dtname',
    right_on='district',
    how='left'
)

# Remove geometry column
final_df = final_df.drop(columns='geometry')

print("\nFinal CSV Ready")


# =========================================================
# 14. EXPORT CSV
# =========================================================

output_csv = "../../district_heat_index_april_2026.csv"

final_df.to_csv(output_csv, index=False)

print(f"\nCSV Saved:\n{output_csv}")